In [1]:
# !pip install keras-tuner

In [1]:
import numpy as np
import pandas as pd
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
import tensorflow as tf
import random

seed = 42

np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

In [3]:
df = pd.read_csv('Data/data_preprocessing.csv', index_col='Date', parse_dates=True)
df.head()

,Precipitation,Temperature,Relative Humidity,Wind Speed
Date,,,,
1985-01-01,0.0,24.12,66.46,26.87
1985-01-02,0.0,24.27,66.08,26.60
1985-01-03,0.1,25.02,61.42,22.44
1985-01-04,0.2,25.26,60.54,22.89
1985-01-05,0.0,25.05,59.92,24.88


In [4]:
# Sắp xếp và kiểm tra dữ liệu
df = df.sort_index()

# kiểm tra liên tục thời gian
print(df.index.to_series().diff().value_counts().head())

Date
1 days    12204
Name: count, dtype: int64


In [5]:
# Chọn biến đầu vào & mục tiêu
features = ['Precipitation', 'Relative Humidity', 'Wind Speed']
target = 'Temperature'

X = df[features]
y = df[target]

In [6]:
# Chia tập dữ liệu: 70% train, 15% validation, 15% test
total_samples = len(X)
train_size = int(total_samples * 0.70)
val_size = int(total_samples * 0.15)

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

In [7]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (8543, 3)
y_train shape: (8543,)
X_val shape: (1830, 3)
y_val shape: (1830,)
X_test shape: (1832, 3)
y_test shape: (1832,)


In [8]:
from sklearn.preprocessing import MinMaxScaler
# Chuẩn hóa dữ liệu
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled   = scaler_X.transform(X_val)
X_test_scaled  = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1,1))
y_val_scaled   = scaler_y.transform(y_val.values.reshape(-1,1))
y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1,1))

In [9]:
# Hàm tạo chuỗi dữ liệu > Dự đoán ngày tiếp theo
def create_sequences(X, y, time_step):
    X_seq, y_seq = [], []
    
    for i in range(len(X) - time_step):
        X_seq.append(X[i:i + time_step])
        y_seq.append(y[i + time_step])
        
    return np.array(X_seq), np.array(y_seq)

In [10]:
# Xác định time_step = 30 
time_step = 30

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, time_step)
X_val_seq, y_val_seq     = create_sequences(X_val_scaled, y_val_scaled, time_step)
X_test_seq, y_test_seq   = create_sequences(X_test_scaled, y_test_scaled, time_step)

In [11]:
# reshape cho LSTM
X_train_seq = X_train_seq.reshape((X_train_seq.shape[0], X_train_seq.shape[1], X_train_seq.shape[2]))
X_val_seq   = X_val_seq.reshape((X_val_seq.shape[0], X_val_seq.shape[1], X_val_seq.shape[2]))

In [12]:
print(X_train_seq.shape)
print(X_val_seq.shape)

(8513, 30, 3)
(1800, 30, 3)


In [13]:
def build_lstm_model(hp):

    model = keras.Sequential()

    model.add(
        layers.LSTM(
            units=hp.Int("lstm1_units", min_value=32, max_value=128, step=32),
            return_sequences=True,
            input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])
        )
    )

    model.add(
        layers.LSTM(
            units=hp.Int("lstm2_units", min_value=32, max_value=128, step=32)
        )
    )

    model.add(
        layers.Dropout(
            rate=hp.Float("dropout_rate", 0.1, 0.5, step=0.1)
        )
    )

    model.add(layers.Dense(1))

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [17]:
# tuner_lstm = kt.RandomSearch(
#     build_lstm,
#     objective="val_loss",
#     max_trials=20,
#     executions_per_trial=1,
#     seed=42,
#     overwrite=True,
#     directory="hyperparameter_tuning",
#     project_name="lstm_search"
# )
tuner_lstm = kt.GridSearch(
    build_lstm_model,
    objective="val_loss",
    executions_per_trial=1,
    seed=42,
    overwrite=True,
    directory="hyperparameter_tuning",
    project_name="lstm_2_grid_search"
)

C:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
tuner_lstm.search(
    X_train_seq,
    y_train_seq,
    epochs=20,
    validation_data=(X_val_seq, y_val_seq),
    batch_size=32,
    shuffle=False
)

Trial 178 Complete [00h 02m 44s]
val_loss: 0.008029455319046974

Best val_loss So Far: 0.007354149594902992
Total elapsed time: 05h 36m 35s

Search: Running Trial #179

Value             |Best Value So Far |Hyperparameter
96                |96                |lstm1_units
128               |128               |lstm2_units
0.5               |0.1               |dropout_rate
0.001             |0.01              |learning_rate

Epoch 1/20


In [ ]:
best_model = tuner_lstm.get_best_models(num_models=1)[0]

In [ ]:
# best_lstm_hp = tuner_lstm.get_best_hyperparameters(num_trials=1)[0]

# print("Best units:", best_lstm_hp.get("units"))
# print("Best learning rate:", best_lstm_hp.get("learning_rate"))

In [ ]:
# best_model_lstm= tuner_lstm.hypermodel.build(best_lstm_hp)
best_hp = tuner_lstm.get_best_hyperparameters(1)[0]
print(best_hp.values)

In [ ]:
best_model_lstm= tuner_lstm.hypermodel.build(best_hp)

In [ ]:
best_model_lstm.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [ ]:
history = best_model_lstm.fit(
    X_train_seq,
    y_train_seq,
    epochs=50,
    validation_data=(X_val_seq, y_val_seq),
    batch_size=32,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
y_pred_scaled = best_model_lstm.predict(X_test_seq)

In [ ]:
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_test_original = scaler_y.inverse_transform(y_test_seq)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse = mean_squared_error(y_test_original, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_original, y_pred)
r2 = r2_score(y_test_original, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
# Vẽ biểu đồ so sánh Thực tế vs Dự đoán
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test_original[:200], label="Actual")
plt.plot(y_pred[:200], label="Predicted")
plt.legend()
plt.title("So sánh nhiệt độ thực tế và dự đoán")
plt.show()

In [ ]:
# Vẽ biểu đồ Loss trong quá trình train
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Training vs Validation Loss")
plt.show()

In [ ]:
mape = np.mean(np.abs((y_test_original - y_pred) / y_test_original)) * 100
print(f"Sai số phần trăm trung bình (MAPE): {mape:.2f}%")

In [ ]:
accuracy = 100 - mape
print(f"Độ chính xác (Accuracy): {accuracy:.2f}%")

In [103]:
# best_model.save("best_lstm2_model.keras")